
# Topological Leaf Pruning vs. PBC-Aware Force Balance

:func:`~pylimer_tools_cpp.Universe.peel_topological_leaves` and
:func:`~pylimer_tools_cpp.Universe.identify_active_atoms` perform an iterative
**2-core peeling** strictly on crosslinker connectivity
(``strand_from`` / ``strand_to``). Because this peeling does **not** inspect
atom coordinates, box dimensions, or periodic boundary conditions (PBC), it
cannot distinguish a loop that wraps across the PBC (elastically active) from
a loop that closes locally inside the box (elastically inactive / solvent).

By contrast, phantom-network force-balance relaxation
(:class:`~pylimer_tools_cpp.MEHPSimulator` with
``assume_box_large_enough=False``) tracks PBC offsets along bonds, so local
loops without entanglements relax to zero tension while PBC-spanning loops
remain stretched and elastically active.

<div class="alert alert-info"><h4>Note</h4>

Retaining local primary loops in the topological 2-core is intentional when
sampling entanglements: even a non-PBC primary loop can become elastically
active once an entanglement or slip-link is associated with it.

</div>


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import pylimer_tools_cpp as pt

BOX_L = 10.0
CROSSLINKER_TYPE = 2
BEAD_TYPE = 1


def build_universe(
    atoms: list[tuple[int, float, float, float]],
    bonds: list[tuple[int, int]],
) -> pt.Universe:
    """Create a periodic Universe of size BOX_L^3 from (type, x, y, z) and bonds."""
    universe = pt.Universe(BOX_L, BOX_L, BOX_L)
    n = len(atoms)
    ids = list(range(1, n + 1))
    types = [a[0] for a in atoms]
    xs = [a[1] for a in atoms]
    ys = [a[2] for a in atoms]
    zs = [a[3] for a in atoms]
    zeros = [0] * n
    universe.add_atoms(ids, types, xs, ys, zs, zeros, zeros, zeros)
    universe.add_bonds([b[0] for b in bonds], [b[1] for b in bonds])
    return universe


# 1. Local primary loop (closes within box, does NOT cross PBC)
u_local_primary = build_universe(
    atoms=[
        (CROSSLINKER_TYPE, 2.0, 2.0, 2.0),
        (BEAD_TYPE, 3.0, 2.0, 2.0),
        (BEAD_TYPE, 3.0, 3.0, 2.0),
        (BEAD_TYPE, 2.0, 3.0, 2.0),
    ],
    bonds=[(1, 2), (2, 3), (3, 4), (4, 1)],
)

# 2. PBC-spanning primary loop (wraps across x = 10 -> 0)
u_pbc_primary = build_universe(
    atoms=[
        (CROSSLINKER_TYPE, 1.0, 5.0, 5.0),
        (BEAD_TYPE, 3.5, 5.0, 5.0),
        (BEAD_TYPE, 6.0, 5.0, 5.0),
        (BEAD_TYPE, 8.5, 5.0, 5.0),
    ],
    bonds=[(1, 2), (2, 3), (3, 4), (4, 1)],
)

# 3. Local 2-crosslinker (secondary) loop (does NOT cross PBC)
u_local_secondary = build_universe(
    atoms=[
        (CROSSLINKER_TYPE, 2.0, 2.0, 2.0),
        (BEAD_TYPE, 3.0, 2.5, 2.0),
        (CROSSLINKER_TYPE, 4.0, 2.0, 2.0),
        (BEAD_TYPE, 3.0, 1.5, 2.0),
    ],
    bonds=[(1, 2), (2, 3), (3, 4), (4, 1)],
)

# 4. PBC-spanning 2-crosslinker loop (one strand direct, one wraps across x)
u_pbc_secondary = build_universe(
    atoms=[
        (CROSSLINKER_TYPE, 1.0, 5.0, 5.0),
        (BEAD_TYPE, 3.0, 5.0, 5.0),
        (CROSSLINKER_TYPE, 5.0, 5.0, 5.0),
        (BEAD_TYPE, 8.0, 5.0, 5.0),
    ],
    bonds=[(1, 2), (2, 3), (3, 4), (4, 1)],
)

# 5. Crosslinker-less solvent ring wrapping across PBC
u_pbc_solvent_ring = build_universe(
    atoms=[
        (BEAD_TYPE, 1.0, 5.0, 5.0),
        (BEAD_TYPE, 3.5, 5.0, 5.0),
        (BEAD_TYPE, 6.0, 5.0, 5.0),
        (BEAD_TYPE, 8.5, 5.0, 5.0),
    ],
    bonds=[(1, 2), (2, 3), (3, 4), (4, 1)],
)

motifs: list[tuple[str, pt.Universe]] = [
    ("Local primary loop\n(non-PBC)", u_local_primary),
    ("PBC primary loop\n(wraps x)", u_pbc_primary),
    ("Local 2-XL loop\n(non-PBC)", u_local_secondary),
    ("PBC 2-XL loop\n(wraps x)", u_pbc_secondary),
    ("PBC ring, no XL\n(all deg-2)", u_pbc_solvent_ring),
]

topo_active_fracs: list[float] = []
mehp_active_fracs: list[float] = []

print(f"{'Motif':<34} | {'Topo 2-core':>11} | {'MEHP (PBC)':>10}")
print("-" * 63)

for label, universe in motifs:
    active_atoms = universe.identify_active_atoms(CROSSLINKER_TYPE)
    topo_frac = float(np.mean(active_atoms))

    sim = pt.MEHPSimulator(
        universe,
        crosslinker_type=CROSSLINKER_TYPE,
        assume_box_large_enough=False,
    )
    sim.run(pt.MEHPLinearSolverConfig())
    mehp_frac = sim.get_active_weight_fraction()

    topo_active_fracs.append(topo_frac)
    mehp_active_fracs.append(mehp_frac)
    flat_label = label.replace("\n", " ")
    print(f"{flat_label:<34} | {topo_frac:>11.2f} | {mehp_frac:>10.2f}")

## Visualizing the Difference
Topological leaf pruning keeps **both** local and PBC-spanning loops active
whenever crosslinkers are present (and prunes crosslinker-less rings even
across PBC). MEHP force-balance relaxation isolates only the PBC-spanning
loops as elastically active.



In [ ]:
x = np.arange(len(motifs))
width = 0.36

fig, ax = plt.subplots(figsize=(9, 4.5))
bars1 = ax.bar(
    x - width / 2,
    topo_active_fracs,
    width,
    label="Topological 2-core (identify_active_atoms)",
)
bars2 = ax.bar(
    x + width / 2,
    mehp_active_fracs,
    width,
    label="MEHP force balance (assume_box_large_enough=False)",
)

ax.set_ylabel("Active atom / weight fraction")
ax.set_title("Topological Leaf Pruning vs. PBC-Aware Force Balance")
ax.set_xticks(x)
ax.set_xticklabels([m[0] for m in motifs])
ax.set_ylim(0.0, 1.2)
ax.legend(loc="upper right")
ax.grid(axis="y", linestyle="--", alpha=0.4)

for bars in (bars1, bars2):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(
            f"{h:.2f}",
            xy=(bar.get_x() + bar.get_width() / 2, h),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
        )

fig.tight_layout()
plt.show()